In [1]:
import pandas as pd
import numpy as np

In [2]:
cities = pd.read_csv('worldcities.csv')

In [3]:
cities.head()

,city,city_ascii,lat,lng,country,iso2,iso3,admin_name,capital,population,id
0,Tokyo,Tokyo,35.6850,139.7514,Japan,JP,JPN,Tōkyō,primary,35676000.0,1392685764
1,New York,New York,40.6943,-73.9249,United States,US,USA,New York,NaN,19354922.0,1840034016
2,Mexico City,Mexico City,19.4424,-99.1310,Mexico,MX,MEX,Ciudad de México,primary,19028000.0,1484247881
3,Mumbai,Mumbai,19.0170,72.8570,India,IN,IND,Mahārāshtra,admin,18978000.0,1356226629
4,São Paulo,Sao Paulo,-23.5587,-46.6250,Brazil,BR,BRA,São Paulo,admin,18845000.0,1076532519


In [4]:
cities.dtypes

city           object
city_ascii     object
lat           float64
lng           float64
country        object
iso2           object
iso3           object
admin_name     object
capital        object
population    float64
id              int64
dtype: object

The following function will be needed to calculate distances:

In [5]:
import math

def haversine_distance(lat1, lon1, lat2, lon2):
    # Radius of the Earth in kilometers
    earth_radius = 6371

    # Convert latitude and longitude from degrees to radians
    lat1 = math.radians(lat1)
    lon1 = math.radians(lon1)
    lat2 = math.radians(lat2)
    lon2 = math.radians(lon2)

    # Haversine formula
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = math.sin(dlat/2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon/2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

    # Calculate the distance
    distance = earth_radius * c

    return distance

if __name__ == "__main__":
    # Coordinates for Antwerp, Belgium
    antwerp_lat, antwerp_lon = 51.2194, 4.4025
    
    # Coordinates for Paris, France
    paris_lat, paris_lon = 48.8566, 2.3522

    distance = haversine_distance(antwerp_lat, antwerp_lon, paris_lat, paris_lon)
    print(f"Distance: {distance:.2f} kilometers")

Distance: 300.75 kilometers


In [29]:
european_cities = ["Paris", "Berlin", "Madrid", "Rome", "Amsterdam", 
                   "Vienna", "Prague", "Barcelona", "Lisbon", "Brussels", 
                   "Budapest", "Warsaw", "Zurich", "Athens", "Copenhagen", 
                   "Stockholm", "Dublin", "Oslo", "Helsinki", "Ljubljana"]

In [34]:
filtered_cities = cities[cities['city_ascii'].isin(european_cities)]

In [35]:
filtered_cities

,city,city_ascii,lat,lng,country,iso2,iso3,admin_name,capital,population,id
19,Paris,Paris,48.8667,2.3333,France,FR,FRA,Île-de-France,primary,9904000.0,1250015082
46,Madrid,Madrid,40.4000,-3.6834,Spain,ES,ESP,Madrid,primary,5567000.0,1724616994
56,Barcelona,Barcelona,41.3833,2.1834,Spain,ES,ESP,Catalonia,admin,4920000.0,1724594040
96,Berlin,Berlin,52.5218,13.4015,Germany,DE,DEU,Berlin,primary,3406000.0,1276451290
98,Rome,Rome,41.8960,12.4833,Italy,IT,ITA,Lazio,primary,3339000.0,1380382862
102,Athens,Athens,37.9833,23.7333,Greece,GR,GRC,Attikí,primary,3242000.0,1300715560
130,Lisbon,Lisbon,38.7227,-9.1449,Portugal,PT,PRT,Lisboa,primary,2812000.0,1620619017
167,Vienna,Vienna,48.2000,16.3666,Austria,AT,AUT,Wien,primary,2400000.0,1040261752
245,Brussels,Brussels,50.8333,4.3333,Belgium,BE,BEL,Brussels-Capital Region,primary,1743000.0,1056469830
254,Warsaw,Warsaw,52.2500,21.0000,Poland,PL,POL,Mazowieckie,primary,1707000.0,1616024847


We get duplicates cities in other continents. Let's remove non_capital-cities.

In [36]:
filtered_cities = cities[(cities['city_ascii'].isin(european_cities))].head(20)
filtered_cities.shape

(20, 11)

In [46]:
filtered_cities = filtered_cities.reset_index(drop=True)

In [47]:
filtered_cities

,city,city_ascii,lat,lng,country,iso2,iso3,admin_name,capital,population,id
0,Paris,Paris,48.8667,2.3333,France,FR,FRA,Île-de-France,primary,9904000.0,1250015082
1,Madrid,Madrid,40.4000,-3.6834,Spain,ES,ESP,Madrid,primary,5567000.0,1724616994
2,Barcelona,Barcelona,41.3833,2.1834,Spain,ES,ESP,Catalonia,admin,4920000.0,1724594040
3,Berlin,Berlin,52.5218,13.4015,Germany,DE,DEU,Berlin,primary,3406000.0,1276451290
4,Rome,Rome,41.8960,12.4833,Italy,IT,ITA,Lazio,primary,3339000.0,1380382862
5,Athens,Athens,37.9833,23.7333,Greece,GR,GRC,Attikí,primary,3242000.0,1300715560
6,Lisbon,Lisbon,38.7227,-9.1449,Portugal,PT,PRT,Lisboa,primary,2812000.0,1620619017
7,Vienna,Vienna,48.2000,16.3666,Austria,AT,AUT,Wien,primary,2400000.0,1040261752
8,Brussels,Brussels,50.8333,4.3333,Belgium,BE,BEL,Brussels-Capital Region,primary,1743000.0,1056469830
9,Warsaw,Warsaw,52.2500,21.0000,Poland,PL,POL,Mazowieckie,primary,1707000.0,1616024847


Creating the distance table

In [51]:
num_cities = len(filtered_cities)
distance_matrix = np.zeros((num_cities, num_cities))

for i in range(num_cities):
    for j in range(num_cities):
        lat1, lon1 = filtered_cities.at[i, 'lat'], filtered_cities.at[i, 'lng']
        lat2, lon2 = filtered_cities.at[j, 'lat'], filtered_cities.at[j, 'lng']
        distance_matrix[i,j] = math.floor(haversine_distance(lat1, lon1, lat2, lon2))

distance_df = pd.DataFrame(distance_matrix, columns=filtered_cities['city'], index=filtered_cities['city'])

print("Distance Matrix:")
print(distance_df)

Distance Matrix:
city         Paris  Madrid  Barcelona  Berlin    Rome  Athens  Lisbon  Vienna  \
city                                                                            
Paris          0.0  1054.0      832.0   877.0  1106.0  2098.0  1453.0  1034.0   
Madrid      1054.0     0.0      505.0  1869.0  1361.0  2368.0   503.0  1808.0   
Barcelona    832.0   505.0        0.0  1499.0   857.0  1877.0  1007.0  1348.0   
Berlin       877.0  1869.0     1499.0     0.0  1183.0  1803.0  2312.0   524.0   
Rome        1106.0  1361.0      857.0  1183.0     0.0  1052.0  1862.0   764.0   
Athens      2098.0  2368.0     1877.0  1803.0  1052.0     0.0  2852.0  1282.0   
Lisbon      1453.0   503.0     1007.0  2312.0  1862.0  2852.0     0.0  2298.0   
Vienna      1034.0  1808.0     1348.0   524.0   764.0  1282.0  2298.0     0.0   
Brussels     261.0  1315.0     1063.0   652.0  1172.0  2089.0  1710.0   915.0   
Warsaw      1366.0  2290.0     1863.0   516.0  1317.0  1600.0  2759.0   557.0   
Budapest   

In [56]:
distance_df.to_csv('distances.csv')

In [57]:
distance_matrix

array([[   0., 1054.,  832.,  877., 1106., 2098., 1453., 1034.,  261.,
        1366., 1248., 1546.,  885., 1908.,  489., 1026.,  777.,  428.,
        1341.,  966.],
       [1054.,    0.,  505., 1869., 1361., 2368.,  503., 1808., 1315.,
        2290., 1976., 2596., 1774., 2949., 1247., 2073., 1451., 1481.,
        2389., 1598.],
       [ 832.,  505.,    0., 1499.,  857., 1877., 1007., 1348., 1063.,
        1863., 1499., 2281., 1354., 2603.,  836., 1759., 1470., 1236.,
        2143., 1117.],
       [ 877., 1869., 1499.,    0., 1183., 1803., 2312.,  524.,  652.,
         516.,  689.,  813.,  281., 1105.,  668.,  355., 1316.,  575.,
         838.,  723.],
       [1106., 1361.,  857., 1183.,    0., 1052., 1862.,  764., 1172.,
        1317.,  812., 1979.,  923., 2202.,  684., 1532., 1883., 1294.,
        2007.,  490.],
       [2098., 2368., 1877., 1803., 1052.,    0., 2852., 1282., 2089.,
        1600., 1123., 2409., 1533., 2469., 1617., 2137., 2853., 2161.,
        2605., 1175.],
       [14

In [58]:
distance_df.index

Index(['Paris', 'Madrid', 'Barcelona', 'Berlin', 'Rome', 'Athens', 'Lisbon',
       'Vienna', 'Brussels', 'Warsaw', 'Budapest', 'Stockholm', 'Prague',
       'Helsinki', 'Zürich', 'Copenhagen', 'Dublin', 'Amsterdam', 'Oslo',
       'Ljubljana'],
      dtype='object', name='city')